In [98]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env from root directory (try parent directory first, then current)
# Use override=True to force reload and overwrite cached values
env_path = Path("../.env") if Path("../.env").exists() else Path(".env")
print(f"Loading from: {env_path.resolve()}")
load_dotenv(env_path, override=True)

print("MODEL_NAME:", os.getenv("LOCAL_MODEL_NAME"))
print("BASE_URL:", os.getenv("LOCAL_MODEL_BASE_URL"))

Loading from: /Users/amoghpednekar/Downloads/EASmartShopping/.env
MODEL_NAME: granite3.1-moe:1b
BASE_URL: http://localhost:11434


## Using Actual LLMs output for Actual output in LLMTestCase of DeepEval

In [99]:
from langchain_ollama import ChatOllama
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env from root directory (try parent directory first, then current)
env_path = Path("../.env") if Path("../.env").exists() else Path(".env")
load_dotenv(env_path, override=True)

# This is the simple LLMs Application to do inferencing
llm = ChatOllama(
    base_url=os.getenv("LOCAL_MODEL_BASE_URL"),
    model=os.getenv("LOCAL_MODEL_NAME"),
    temperature=0.7,
    reasoning=False
)

In [100]:
!deepeval set-ollama --model granite3.1-moe:1b

🙌 Congratulations! You're now using a local Ollama model `granite3.1-moe:1b` 
for all evals that require an LLM.


### Login to Confident AI

In [ ]:
import deepeval
import os
from pathlib import Path
from dotenv import load_dotenv

# Ensure .env is loaded
env_path = Path("../.env") if Path("../.env").exists() else Path(".env")
load_dotenv(env_path, override=True)

# Load API key from environment (keep it safe, not hardcoded!)
api_key = os.getenv("CONFIDENT_AI_API_KEY")
if api_key:
    deepeval.login(api_key=api_key)
else:
    print("⚠️  Warning: CONFIDENT_AI_API_KEY not found in .env. Skipping Confident AI login.")
    print("   Optional: Add it to .env if you want to track results on Confident AI.")


🎉🥳 Congratulations! You've successfully logged in! 🙌

## Application chat mode interface

In [102]:
BACKEND_URL = os.environ.get("BACKEND_URL", "http://localhost:8000")
import requests
import json

def chat(message: str, session_id: str) -> str:
    """
    Call the streaming chat endpoint and collect the full assistant response.
    Returns the concatenated text content.
    """
    resp = requests.post(
        f"{BACKEND_URL}/api/chat/stream",
        json={"message": message, "session_id": session_id},
        stream=True,
        timeout=120,
    )
    resp.raise_for_status()

    full_text = ""
    for line in resp.iter_lines():
        if not line:
            continue
        decoded = line.decode("utf-8") if isinstance(line, bytes) else line
        if not decoded.startswith("data: "):
            continue
        payload = decoded[6:].strip()
        if not payload:
            continue
        try:
            data = json.loads(payload)
            if data.get("type") == "chunk":
                full_text += data.get("content", "")
        except json.JSONDecodeError:
            pass
    return full_text.strip()


## Session ID creation

In [103]:
import uuid
def new_session() -> str:
    """
    Generate a new session ID.
    """
    return f"test_{uuid.uuid4().hex[:12]}"

In [104]:
session = new_session()

In [105]:
session

'test_497c4af19094'

In [106]:
response = chat("add trail running shoes size UK 9 color black/white to my cart", session)
print(response)

I have successfully added the TerraTrek XT Trail Running Shoes in size UK 9, color Black/White to your cart for $109.99.

Is there anything else I can assist you with today?


## Get Cart details

In [107]:
def get_cart(session_id: str) -> dict:
    resp = requests.get(f"{BACKEND_URL}/api/cart/{session_id}", timeout=10)
    return resp.json()

In [108]:
cart_response = get_cart(session)
print(cart_response)

{'session_id': 'test_497c4af19094', 'items': [{'product_id': 10, 'quantity': 1, 'selected_options': {'Color': 'Black/White', 'Size': 'UK 9'}, 'id': 17, 'product': {'name': 'TerraTrek XT Trail Running Shoes', 'description': 'Aggressive 5mm multi-directional lugged outsole for off-road grip on mud, gravel and rock. Protective rock-plate in forefoot, reinforced toe cap, Gore-Tex waterproof membrane. Rated 4.8/5 from 976 trail runners — top pick for durability on technical terrain. Outsole rated 600km off-road. Manufacturer: TerraTrek.', 'price': 109.99, 'category': 'Sports', 'image_url': '🥾', 'stock': 90, 'options': [{'name': 'Size', 'values': ['UK 6', 'UK 7', 'UK 8', 'UK 9', 'UK 10', 'UK 11', 'UK 12']}, {'name': 'Color', 'values': ['Black/White', 'All White', 'Navy/Red', 'Gray/Lime']}], 'rating': 0.0, 'review_count': 0, 'specs': {}, 'manufacturer': '', 'id': 10, 'created_at': '2026-07-01 07:51:39'}}], 'total': 109.99}


## Test Code to check the chat response with model we have

In [109]:
from deepeval.test_case import LLMTestCase, LLMTestCaseParams;
from deepeval import evaluate;
from deepeval.metrics import GEval
from pathlib import Path
from dotenv import load_dotenv
from deepeval.evaluate import AsyncConfig
from deepeval.models import OllamaModel
import os
from typing import Optional, Tuple, Union
from pydantic import BaseModel

# Load .env from root directory (try parent directory first, then current)
env_path = Path("../.env") if Path("../.env").exists() else Path(".env")
load_dotenv(env_path, override=True)


class OllamaModelNoThink(OllamaModel):
     def generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model()
        messages = [{"role": "user", "content": prompt}]

        response = chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

     async def a_generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model(async_mode=True)
        messages = [{"role": "user", "content": prompt}]

        response = await chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

ollama_model = OllamaModelNoThink(
    model=os.getenv("LOCAL_MODEL_NAME"), 
    base_url=os.getenv("LOCAL_MODEL_BASE_URL")
)

session = new_session()
chat_response = chat("add trail running shoes size UK 9 color black/white to my cart", session)
cart_details = get_cart(session)

cart_response = json.dumps(cart_details, indent=2)

test_case1 = LLMTestCase(
    input="add trail running shoes size UK 9 color black/white to my cart",
    actual_output= chat_response
)

confirmation = GEval(
    name = "add to cart confirmation",
    criteria = (
        
        "The assistant should confirm that the item has been successfully added to the cart or not"
        "or ask user to select the size or color if not provided"
        "score high for clear confirmation messages or prompt options and low for vague or missing confirmations."
    ),
    evaluation_params = [LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold = 0.5,
    model = ollama_model
)

test_case2 = LLMTestCase(
    input="add trail running shoes size UK 9 color black/white to my cart",
    actual_output= cart_response
)

correctness = GEval(
    name = "check cart item correctnees",
    criteria = (
        
        "check whether items in the cart corresponds to trail running shoes"
        "score high if cart contains items as requested by user or score low if it doesn't"
    ),
    evaluation_steps=[
        "check the cart contains atleast one item",
        "check whether the product name or description matches trail running shoes"

    ],
    evaluation_params = [LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold = 0.5,
    model = ollama_model
)

evaluate(test_cases =[test_case1, test_case2],
         metrics = [correctness,confirmation],
         async_config=AsyncConfig(run_async=False)
)

✨ You're running DeepEval's latest check cart item correctnees [GEval] Metric! (using granite3.1-moe:1b (Ollama), 
strict=False, async_mode=False)...

✨ You're running DeepEval's latest add to cart confirmation [GEval] Metric! (using granite3.1-moe:1b (Ollama), 
strict=False, async_mode=False)...



Metrics Summary

  - ✅ check cart item correctnees [GEval] (score: 0.9, threshold: 0.5, strict: False, evaluation model: granite3.1-moe:1b (Ollama), reason: The response correctly checks if the cart contains at least one item, which aligns with step 1 of the evaluation. It also accurately identifies that the product name matches 'trail running shoes', which is a key requirement for this task., error: None)
  - ✅ add to cart confirmation [GEval] (score: 1.0, threshold: 0.5, strict: False, evaluation model: granite3.1-moe:1b (Ollama), reason: The response correctly identifies that the item has been added to the cart, as indicated by the confirmation message. It also accurately states that the user is now on the 'My Cart' page., error: None)

For test case:

  - input: add trail running shoes size UK 9 color black/white to my cart
  - actual output: I have successfully added the TerraTrek XT Trail Running Shoes in size UK 9, color Black/White to your cart for $109.99.

Is there anything

⚠ WARNING: No hyperparameters logged.
» ]8;id=10465800;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=10465804;https://app.confident-ai.com/project/cmqqngl3w001cqv13wbs679kw/test-runs/cmrnleddp00pukk13c19qfms1/regression-testing\https://app.confident-ai.com/project/cmqqngl3w001cqv13wbs679kw/test-runs/cmrnleddp00pukk13c19qfms1/regression-testi]8;;\
]8;id=10465804;https://app.confident-ai.com/project/cmqqngl3w001cqv13wbs679kw/test-runs/cmrnleddp00pukk13c19qfms1/regression-testing\ng]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='check cart item correctnees [GEval]', threshold=0.5, success=True, score=0.9, reason="The response correctly checks if the cart contains at least one item, which aligns with step 1 of the evaluation. It also accurately identifies that the product name matches 'trail running shoes', which is a key requirement for this task.", strict_mode=False, evaluation_model='granite3.1-moe:1b (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Criteria:\ncheck whether items in the cart corresponds to trail running shoesscore high if cart contains items as requested by user or score low if it doesn\'t \n \nEvaluation Steps:\n[\n    "check the cart contains atleast one item",\n    "check whether the product name or description matches trail running shoes"\n] \n \nRubric:\nNone \n \nScore: 0.9'), MetricData(name='add to cart confirmation [GEval]', threshold=0.5, success=True, score=1.0, reas